# ST-GCN: Обучение классификатора жестов РЖЯ (Slovo, 1000 классов)

**Google Colab Pro** | GPU: A100 / H100 | Время: ~8–12 ч

## Что делает этот ноутбук
1. Загружает предобработанный датасет Slovo RSL из Google Drive (numpy-массивы)
2. Определяет и обучает **ST-GCN** (Spatial-Temporal Graph Convolutional Network)
   на 75 ключевых точках MediaPipe (33 поза + 21 левая рука + 21 правая рука)
3. Логирует метрики в **MLflow на DAGsHub** (`https://dagshub.com/noviyblock/glossa`)
4. Экспортирует ONNX-модель в формате `(B, T, 75, 3)` → `(B, 1000)`
5. Пушит модель через DVC

## Перед запуском
Добавьте в **Colab → Secrets (🔑)**:
- `DAGSHUB_TOKEN` — токен DAGsHub (Settings → Access Tokens)
- `MLFLOW_TRACKING_USERNAME` — ваш логин DAGsHub (`noviyblock`)
- `MLFLOW_TRACKING_PASSWORD` — тот же токен DAGsHub

Датасет должен быть в Drive по пути `/content/drive/MyDrive/glossa/data/`:
```
data/gestures/processed/train/features.npy  # (N, 64, 75, 3)
data/gestures/processed/train/labels.npy
data/gestures/processed/val/features.npy
data/gestures/processed/val/labels.npy
data/gestures/processed/class_names.json
```

In [ ]:
# ── 1. Установка зависимостей ─────────────────────────────────────────────────
!pip install -q dagshub mlflow pyyaml onnx onnxruntime
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
print('Done')

In [ ]:
# ── 2. Google Drive + Colab Secrets + DAGsHub ────────────────────────────────
import os

# Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
DRIVE_ROOT = '/content/drive/MyDrive/glossa'
DATA_DIR   = f'{DRIVE_ROOT}/data'
MODELS_DIR = f'{DRIVE_ROOT}/models'
os.makedirs(MODELS_DIR, exist_ok=True)

# Colab Secrets
from google.colab import userdata
for key in ('DAGSHUB_TOKEN', 'MLFLOW_TRACKING_USERNAME', 'MLFLOW_TRACKING_PASSWORD'):
    try:
        os.environ[key] = userdata.get(key)
    except Exception:
        print(f'[warn] Secret {key} not found — MLflow logging may be disabled')

token = os.environ.get('DAGSHUB_TOKEN') or os.environ.get('MLFLOW_TRACKING_PASSWORD', '')
if token:
    os.environ.setdefault('AWS_ACCESS_KEY_ID', token)
    os.environ.setdefault('AWS_SECRET_ACCESS_KEY', token)
    os.environ.setdefault('MLFLOW_S3_ENDPOINT_URL', 'https://dagshub.com/noviyblock/glossa.s3')

# DAGsHub MLflow
try:
    import dagshub
    dagshub.init(repo_owner='noviyblock', repo_name='glossa', mlflow=True)
    print('[DAGsHub] dagshub.init() OK')
    print('[DAGsHub] UI: https://dagshub.com/noviyblock/glossa')
except Exception as e:
    import mlflow
    mlflow.set_tracking_uri('https://dagshub.com/noviyblock/glossa.mlflow')
    print(f'[MLflow] fallback: {e}')

In [ ]:
# ── 3. Конфигурация ──────────────────────────────────────────────────────────
import time

CFG = {
    # Data
    'num_classes':    1000,
    'sequence_length': 64,
    'num_nodes':       75,
    # Training
    'batch_size':     32,
    'epochs':         100,
    'lr':             1e-3,
    'weight_decay':   1e-4,
    'patience':       10,
    'flip_prob':      0.5,
    'seed':           42,
    # ONNX
    'opset_version':  17,
    # MLflow
    'experiment_name': 'gesture_stgcn_slovo',
    'run_name':        f'stgcn_train_{time.strftime("%Y%m%d_%H%M")}',
}

import random
import numpy as np
import torch

random.seed(CFG['seed'])
np.random.seed(CFG['seed'])
torch.manual_seed(CFG['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG['seed'])

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── 4. Граф MediaPipe Holistic (75 узлов) ────────────────────────────────────
import numpy as np

_POSE_EDGES = [
    (0,1),(1,2),(2,3),(3,7),(0,4),(4,5),(5,6),(6,8),(9,10),
    (11,12),(11,13),(13,15),(12,14),(14,16),
    (15,17),(15,19),(15,21),(17,19),(16,18),(16,20),(16,22),(18,20),
    (11,23),(12,24),(23,24),(23,25),(24,26),(25,27),(26,28),
    (27,29),(27,31),(28,30),(28,32),(29,31),(30,32),
]
_HAND_EDGES = [
    (0,1),(1,2),(2,3),(3,4),
    (0,5),(5,6),(6,7),(7,8),
    (0,9),(9,10),(10,11),(11,12),
    (0,13),(13,14),(14,15),(15,16),
    (0,17),(17,18),(18,19),(19,20),
    (5,9),(9,13),(13,17),(5,17),
]
L_OFF, R_OFF, N = 33, 54, 75


def build_adjacency(num_nodes: int = N) -> np.ndarray:
    """D^{-1/2} (A+I) D^{-1/2} normalised adjacency, shape (75, 75)."""
    A = np.eye(num_nodes, dtype=np.float32)
    for i, j in _POSE_EDGES:
        A[i, j] = A[j, i] = 1.0
    for i, j in _HAND_EDGES:
        A[i+L_OFF, j+L_OFF] = A[j+L_OFF, i+L_OFF] = 1.0
        A[i+R_OFF, j+R_OFF] = A[j+R_OFF, i+R_OFF] = 1.0
    A[15, L_OFF] = A[L_OFF, 15] = 1.0   # left wrist → hand root
    A[16, R_OFF] = A[R_OFF, 16] = 1.0   # right wrist → hand root
    deg = A.sum(1)
    d = np.where(deg > 0, 1.0 / np.sqrt(deg), 0.0)
    return (d[:, None] * A * d[None, :]).astype(np.float32)


A_np = build_adjacency()
print(f'Adjacency shape: {A_np.shape}, density: {(A_np > 0).mean():.3f}')

In [ ]:
# ── 5. Архитектура ST-GCN ────────────────────────────────────────────────────
import torch
import torch.nn as nn

A_GLOBAL = torch.tensor(A_np).to(DEVICE)


class GraphConv(nn.Module):
    """Single-partition GCN: W(x) @ A (ONNX-compatible via einsum)."""
    def __init__(self, in_ch: int, out_ch: int) -> None:
        super().__init__()
        self.register_buffer('A', A_GLOBAL.clone())
        self.fc = nn.Conv2d(in_ch, out_ch, 1)
        self.bn = nn.BatchNorm2d(out_ch)

    def forward(self, x):  # (B, C, T, V)
        x = self.fc(x)
        x = torch.einsum('bctv,vw->bctw', x, self.A)
        return self.bn(x)


class TCN(nn.Module):
    def __init__(self, ch: int, stride: int = 1, kernel: int = 9) -> None:
        super().__init__()
        pad = (kernel - 1) // 2
        self.net = nn.Sequential(
            nn.Conv2d(ch, ch, (kernel, 1), stride=(stride, 1), padding=(pad, 0)),
            nn.BatchNorm2d(ch),
        )

    def forward(self, x):
        return self.net(x)


class STGCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, residual=True):
        super().__init__()
        self.gcn = GraphConv(in_ch, out_ch)
        self.tcn = TCN(out_ch, stride=stride)
        self.relu = nn.ReLU(inplace=True)
        self._no_res = not residual
        if residual and (in_ch != out_ch or stride != 1):
            self.skip = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=(stride, 1)),
                nn.BatchNorm2d(out_ch),
            )
        elif residual:
            self.skip = nn.Identity()
        else:
            self.skip = None

    def forward(self, x):
        out = self.tcn(self.gcn(x))
        res = 0 if self._no_res else self.skip(x)
        return self.relu(out + res)


class STGCN(nn.Module):
    """
    ST-GCN для RSL (Slovo): 1000 классов, 75 узлов MediaPipe.
    Вход:  (B, T, 75, 3)
    Выход: (B, 1000)
    """
    _CFG = [
        (3,   64,  1, False),
        (64,  64,  1, True),
        (64,  64,  1, True),
        (64,  64,  1, True),
        (64,  128, 2, True),
        (128, 128, 1, True),
        (128, 128, 1, True),
        (128, 256, 2, True),
        (256, 256, 1, True),
        (256, 256, 1, True),
    ]

    def __init__(self, num_classes: int = 1000, num_nodes: int = 75) -> None:
        super().__init__()
        self.num_nodes = num_nodes
        self.data_bn = nn.BatchNorm1d(3 * num_nodes)
        self.layers = nn.ModuleList(
            [STGCNBlock(ic, oc, st, res) for ic, oc, st, res in self._CFG]
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.drop = nn.Dropout(0.5)
        self.fc   = nn.Linear(256, num_classes)

    def forward(self, x):  # x: (B, T, V, C)
        B, T, V, C = x.shape
        x = x.permute(0, 1, 3, 2).contiguous().view(B * T, C * V)
        x = self.data_bn(x)
        x = x.view(B, T, C, V).permute(0, 2, 1, 3).contiguous()  # (B, C, T, V)
        for layer in self.layers:
            x = layer(x)
        x = self.pool(x).view(B, -1)
        return self.fc(self.drop(x))


model = STGCN(num_classes=CFG['num_classes'], num_nodes=CFG['num_nodes']).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'ST-GCN параметры: {n_params:,} ({n_params/1e6:.1f}M)')

# Smoke-test
with torch.no_grad():
    dummy = torch.randn(2, CFG['sequence_length'], CFG['num_nodes'], 3).to(DEVICE)
    out = model(dummy)
    print(f'Smoke-test: вход {tuple(dummy.shape)} → выход {tuple(out.shape)}')

In [ ]:
# ── 6. Загрузка данных ────────────────────────────────────────────────────────
import json
from pathlib import Path
from torch.utils.data import TensorDataset, DataLoader

PROCESSED = Path(DATA_DIR) / 'gestures' / 'processed'


def load_split(split: str):
    d = PROCESSED / split
    X = np.load(d / 'features.npy').astype(np.float32)
    y = np.load(d / 'labels.npy').astype(np.int64)
    # Normalise shape from legacy (N,T,225) to (N,T,75,3) if needed
    if X.ndim == 3:
        X = X.reshape(X.shape[0], X.shape[1], 75, 3)
    print(f'[{split}] X={X.shape}  y={y.shape}  classes={np.unique(y).size}')
    return X, y


X_train, y_train = load_split('train')
X_val,   y_val   = load_split('val')

class_names_path = PROCESSED / 'class_names.json'
class_names = json.loads(class_names_path.read_text()) if class_names_path.exists() else []
print(f'Классов: {len(class_names)}')

# Augmentation: горизонтальный флип (зеркальное отражение)
# Поменять местами левую и правую стороны позы + руки
_POSE_FLIP = [
    0,2,1,4,3,6,5,8,7,10,9,12,11,14,13,16,15,18,17,20,19,22,21,24,23,26,25,28,27,30,29,32,31
]
_LEFT_IDX  = list(range(33, 54))
_RIGHT_IDX = list(range(54, 75))


def horizontal_flip(x):  # x: (T, 75, 3)
    flipped = x.copy()
    # Flip x-axis
    flipped[:, :, 0] = -flipped[:, :, 0]
    # Swap left/right pose joints
    flipped[:, :33] = flipped[:, _POSE_FLIP]
    # Swap hands
    tmp = flipped[:, _LEFT_IDX].copy()
    flipped[:, _LEFT_IDX]  = flipped[:, _RIGHT_IDX]
    flipped[:, _RIGHT_IDX] = tmp
    return flipped


# Apply flip augmentation to training set (double the data)
flip_mask = np.random.rand(len(X_train)) < CFG['flip_prob']
X_aug = X_train.copy()
for i in np.where(flip_mask)[0]:
    X_aug[i] = horizontal_flip(X_aug[i])
print(f'Augmented {flip_mask.sum()} / {len(X_train)} samples')

train_ds = TensorDataset(
    torch.tensor(X_aug),
    torch.tensor(y_train),
)
train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,
                          num_workers=2, pin_memory=True)

X_v = torch.tensor(X_val).to(DEVICE)
y_v = torch.tensor(y_val)
print(f'Train batches: {len(train_loader)}')

In [ ]:
# ── 7. Обучение с MLflow ─────────────────────────────────────────────────────
import mlflow

EXPERIMENT_NAME = CFG['experiment_name']
mlflow.set_experiment(EXPERIMENT_NAME)

optimizer = torch.optim.Adam(
    model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay']
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG['epochs']
)
criterion = nn.CrossEntropyLoss()

best_val_acc = 0.0
best_val_f1  = 0.0
no_improve   = 0
CKPT_BEST    = f'{MODELS_DIR}/stgcn_best.pt'
CKPT_FINAL   = f'{MODELS_DIR}/stgcn_final.pt'

with mlflow.start_run(run_name=CFG['run_name']) as run:
    RUN_ID = run.info.run_id
    print(f'MLflow run: {RUN_ID}')

    mlflow.log_params({
        'architecture':    'STGCN',
        'num_classes':     CFG['num_classes'],
        'num_nodes':       CFG['num_nodes'],
        'sequence_length': CFG['sequence_length'],
        'batch_size':      CFG['batch_size'],
        'lr':              CFG['lr'],
        'weight_decay':    CFG['weight_decay'],
        'epochs':          CFG['epochs'],
        'patience':        CFG['patience'],
        'dataset':         'slovo_rsl_1000',
        'gpu':             torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
    })

    for epoch in range(1, CFG['epochs'] + 1):
        # ── Train ──────────────────────────────────────────────────
        model.train()
        epoch_loss = 0.0
        correct = 0
        total   = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            logits = model(xb)
            loss   = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            correct += (logits.argmax(1) == yb).sum().item()
            total   += len(yb)
        scheduler.step()

        train_loss = epoch_loss / len(train_loader)
        train_acc  = correct / total

        # ── Validate ───────────────────────────────────────────────
        model.eval()
        with torch.no_grad():
            val_logits = model(X_v)
            val_pred   = val_logits.argmax(1).cpu()

        val_acc  = (val_pred == y_v).float().mean().item()

        # Top-5 accuracy
        top5_pred = val_logits.topk(5, dim=1).indices.cpu()
        top5_acc  = (top5_pred == y_v.unsqueeze(1)).any(dim=1).float().mean().item()

        # Simple per-class F1 (macro)
        from sklearn.metrics import f1_score
        val_f1 = f1_score(y_v.numpy(), val_pred.numpy(), average='macro', zero_division=0)

        mlflow.log_metrics({
            'train_loss': train_loss,
            'train_acc':  train_acc,
            'val_acc':    val_acc,
            'val_top5':   top5_acc,
            'val_f1_macro': val_f1,
            'lr': scheduler.get_last_lr()[0],
        }, step=epoch)

        if epoch % 5 == 0 or epoch == 1:
            print(f'Epoch {epoch:3d}/{CFG["epochs"]}  '
                  f'loss={train_loss:.4f}  acc={train_acc:.3f}  '
                  f'val_acc={val_acc:.3f}  val_f1={val_f1:.3f}')

        if val_f1 > best_val_f1:
            best_val_f1  = val_f1
            best_val_acc = val_acc
            no_improve   = 0
            torch.save(model.state_dict(), CKPT_BEST)
        else:
            no_improve += 1
            if no_improve >= CFG['patience']:
                print(f'Early stopping at epoch {epoch}')
                break

    torch.save(model.state_dict(), CKPT_FINAL)
    mlflow.log_artifact(CKPT_BEST,  'checkpoints')
    mlflow.log_artifact(CKPT_FINAL, 'checkpoints')
    mlflow.log_metrics({
        'best_val_acc': best_val_acc,
        'best_val_f1':  best_val_f1,
    })
    mlflow.set_tag('status', 'trained')
    print(f'\nОбучение завершено. best_val_acc={best_val_acc:.4f}  best_val_f1={best_val_f1:.4f}')
    print(f'MLflow run: https://dagshub.com/noviyblock/glossa.mlflow/#/experiments/0/runs/{RUN_ID}')

In [ ]:
# ── 8. Экспорт ONNX ──────────────────────────────────────────────────────────
ONNX_PATH = f'{MODELS_DIR}/gesture_classifier.onnx'

# Загружаем лучший чекпоинт
model.load_state_dict(torch.load(CKPT_BEST, map_location=DEVICE))
model.eval()

dummy = torch.randn(1, CFG['sequence_length'], CFG['num_nodes'], 3).to(DEVICE)
torch.onnx.export(
    model,
    dummy,
    ONNX_PATH,
    export_params=True,
    opset_version=CFG['opset_version'],
    input_names=['keypoints'],
    output_names=['logits'],
    dynamic_axes={
        'keypoints': {0: 'batch_size', 1: 'num_frames'},
        'logits':    {0: 'batch_size'},
    },
)
print(f'ONNX экспортирован: {ONNX_PATH}')

# Verify
import onnxruntime as ort
sess = ort.InferenceSession(ONNX_PATH, providers=['CPUExecutionProvider'])
inp  = dummy.cpu().numpy()
out  = sess.run(['logits'], {'keypoints': inp})[0]
print(f'ONNX verify: вход {inp.shape} → выход {out.shape}')
print(f'Top-1 класс (случайный вход): {out.argmax()}')

# Логируем в MLflow
with mlflow.start_run(run_id=RUN_ID):
    mlflow.log_artifact(ONNX_PATH, 'onnx')
print('ONNX залогирован в MLflow')

In [ ]:
# ── 9. DVC push (опционально) ─────────────────────────────────────────────────
# Скопируем ONNX в корень репозитория и запушим через DVC
import subprocess

REPO_DIR = '/content/glossa'  # если репо клонировано в /content

if not os.path.exists(REPO_DIR):
    # Клонирование репозитория (если ещё не клонировано)
    subprocess.run([
        'git', 'clone', 'https://github.com/noviyblock/glossa.git', REPO_DIR
    ], check=True)

# Копируем модель
import shutil
target = f'{REPO_DIR}/models/gesture_classifier.onnx'
os.makedirs(os.path.dirname(target), exist_ok=True)
shutil.copy2(ONNX_PATH, target)
print(f'Скопировано: {target}')

# DVC add + push
for cmd in [
    ['dvc', 'add', 'models/gesture_classifier.onnx'],
    ['dvc', 'push'],
]:
    result = subprocess.run(cmd, cwd=REPO_DIR, capture_output=True, text=True)
    print(' '.join(cmd), '→', result.returncode)
    if result.stdout: print(result.stdout)
    if result.stderr: print(result.stderr)

print('\nГотово!')
print(f'best_val_acc = {best_val_acc:.4f}')
print(f'best_val_f1  = {best_val_f1:.4f}')
print(f'MLflow UI: https://dagshub.com/noviyblock/glossa.mlflow')

## Результаты и следующие шаги

После обучения:
1. Скачайте `stgcn_best.pt` и `gesture_classifier.onnx` из Drive
2. Разместите `gesture_classifier.onnx` в `models/` репозитория
3. Запустите `dvc repro exp_01_gesture_backbone` для регистрации метрик
4. Откройте MLflow UI: https://dagshub.com/noviyblock/glossa.mlflow

**Целевые метрики (из params.yaml):**
- `val_acc` ≥ 0.90
- `val_f1_macro` ≥ 0.88
- P95 latency ≤ 50 мс (проверяется в exp 03)